# Phase 6: Explainability and Attribution
    
> **Execution context:** This notebook runs on Google Colab or Kaggle.
> **Scope:** This phase uses post-hoc interpretability methods (SHAP, Integrated Gradients) to explain the predictions of the frozen ML and GNN models previously trained (XGBoost from Phase 2, GraphSAGE from Phase 3, HeteroRGCN from Phases 4/5).
> **Constraint:** It introduces NO new models or architectural changes.

## Objective
Answer the fundamental question: *Why does BitcoinGraphGuard make its predictions?*
Understand XGBoost tabular features, GraphSAGE homogeneous structure reliance, and HeteroRGCN heterogeneous relationship exploitation. Ensure explanations adhere to the strict temporal boundary rules (explanations at step $t$ use only context available at $\le t$).

In [ ]:
# 1. Environment & Setup
import os
import sys
import gc
import json
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch import nn

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = "/content/drive/MyDrive/Projects/Bitcoin Graph"
    data_dir = os.path.join(REPO_ROOT, "Og data")
elif IN_KAGGLE:
    REPO_ROOT = "/kaggle/working/BitcoinGraphGuard"
    os.makedirs(REPO_ROOT, exist_ok=True)
    data_dir = "/kaggle/input/elliptic-data-set/elliptic_bitcoin_dataset"
else:
    REPO_ROOT = os.path.abspath("..")
    data_dir = os.path.join(REPO_ROOT, "Og data")

# Phase artifacts live under results/ in the repository; fall back to the Drive-root layout.
RESULTS_DIR = os.path.join(REPO_ROOT, "results")
if not os.path.isdir(RESULTS_DIR):
    RESULTS_DIR = REPO_ROOT

for label, path in [
    ("repository", REPO_ROOT),
    ("raw data (Og data)", data_dir),
    ("results", RESULTS_DIR),
]:
    if not os.path.isdir(path):
        raise FileNotFoundError(f"{label} directory not found: {path}")

OUT_DIR = os.path.join(RESULTS_DIR, "explainability")
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)

# shap and captum are Phase-6 extras (not in requirements.txt); install them on demand.
for _module, _package in [("shap", "shap"), ("captum", "captum")]:
    try:
        __import__(_module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _package])

print(f"Repository Root: {REPO_ROOT}")
print(f"Data Dir: {data_dir}")
print(f"Results Dir: {RESULTS_DIR}")
print(f"Output Dir: {OUT_DIR}")

## 2. Load Core Data Maps and Predictions
Load node mappings and validation/test temporal splits.

In [ ]:
# Node universe, labels and frozen predictions. txs_features.csv HAS a header row, so it must be
# read with header=0 (header=None would push the literal string "Time step" into the dtype).
classes_df = pd.read_csv(os.path.join(data_dir, "txs_classes.csv"), dtype={"txId": "int64"})
features_df = pd.read_csv(
    os.path.join(data_dir, "txs_features.csv"), usecols=["txId", "Time step"]
)
features_df.rename(columns={"Time step": "time_step"}, inplace=True)
features_df["time_step"] = features_df["time_step"].astype("int16")

node_to_idx = {tx: i for i, tx in enumerate(features_df["txId"].values)}
idx_to_node = {i: tx for tx, i in node_to_idx.items()}
time_steps = features_df["time_step"].to_numpy()

# Labels: 1=illicit -> 1, 2=licit -> 0, 3=unknown -> 2. Unknown is never a training example.
classes_df["label"] = classes_df["class"].map({1: 1, 2: 0, 3: 2})
y_all = classes_df["label"].to_numpy()

xgb_preds = pd.read_csv(os.path.join(RESULTS_DIR, "xgboost", "predictions.csv")).set_index("txId")
gs_preds = pd.read_csv(os.path.join(RESULTS_DIR, "GraphSage", "predictions.csv")).set_index("txId")
rgcn_preds = pd.read_csv(
    os.path.join(RESULTS_DIR, "temporal_inductive", "rgcn_temporal_predictions.csv")
)
rgcn_preds["rgcn_score"] = rgcn_preds["static_score"]  # Phase-5 static model = frozen RGCN
rgcn_preds = rgcn_preds.set_index("txId")

print(f"Total nodes: {len(features_df):,} | labeled: {int((y_all != 2).sum()):,}")
print(f"XGBoost test predictions  : {len(xgb_preds)}")
print(f"GraphSAGE test predictions: {len(gs_preds)}")
print(f"RGCN (Phase 5 static)     : {len(rgcn_preds)}")

## 3. XGBoost Explanations (SHAP)
Extensively analyse XGBoost's tabular logic. XGBoost achieved a strict PR-AUC ceiling (0.8013) on purely handcrafted tabular aggregates.

In [ ]:
import shap
import xgboost as xgb

with open(os.path.join(RESULTS_DIR, "xgboost", "selected_features.json")) as f:
    xgb_meta = json.load(f)
xgb_features = list(xgb_meta["features"])  # 165 Local_/Aggregate_ columns
XGB_THRESHOLD = 0.435  # frozen Phase-2 operating point

# Stratify the explanation sample by TP / FP / FN / TN in the early test window (35-42), using
# the XGBoost score (the notebook previously filtered on the logistic-regression score).
xgb_early = xgb_preds[(xgb_preds["time_step"] >= 35) & (xgb_preds["time_step"] <= 42)]
score = xgb_early["xgboost_score"]
quadrants = {
    "tp": xgb_early[(xgb_early["y_true"] == 1) & (score >= XGB_THRESHOLD)],
    "fp": xgb_early[(xgb_early["y_true"] == 0) & (score >= XGB_THRESHOLD)],
    "fn": xgb_early[(xgb_early["y_true"] == 1) & (score < XGB_THRESHOLD)],
    "tn": xgb_early[(xgb_early["y_true"] == 0) & (score < XGB_THRESHOLD)],
}
rng = np.random.default_rng(42)
shap_sample_idx = np.concatenate(
    [rng.choice(q.index.to_numpy(), size=min(100, len(q)), replace=False) for q in quadrants.values()]
)
print("quadrant counts:", {k: len(v) for k, v in quadrants.items()}, "-> sample", shap_sample_idx.size)

# Read only the 165 model columns for the sampled rows (the raw CSV is ~700 MB).
tx_features = pd.read_csv(
    os.path.join(data_dir, "txs_features.csv"), usecols=["txId"] + xgb_features
).set_index("txId")
X_shap = tx_features.loc[shap_sample_idx, xgb_features].astype("float32")

# Frozen optimised XGBoost booster. Trees are scale-invariant, so the Phase-2 scaler (used only
# by the logistic-regression pipeline) is NOT applied here.
booster = xgb.Booster()
booster.load_model(os.path.join(RESULTS_DIR, "xgboost", "xgb_model_optimized.json"))
assert booster.num_features() == len(xgb_features), "model/feature-name mismatch"

explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_shap)

fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=xgb_features, max_display=20, show=False)
plt.title("XGBoost Global SHAP Summary (Top 20 Features, Steps 35-42)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figures", "xgb_shap_summary.png"), dpi=150)
plt.show()

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = (
    pd.DataFrame({"feature": xgb_features, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_importance.to_csv(os.path.join(OUT_DIR, "xgb_shap_global.csv"), index=False)
print(shap_importance.head(5).to_string(index=False))

## 4. PyTorch GNN Architecture Reloading
Load the exact GraphSAGE and RGCN definitions matching the saved `state_dict`s.

In [ ]:
# Restored verbatim from Phase 3 (notebook 03) and Phase 4/5 (notebooks 04/05) so the reloaded
# classes are exactly the ones the checkpoints were saved from. The previous draft reloaded a
# PyG HeteroConv model whose parameter names could never match the saved Phase-4 checkpoint.
try:
    from torch_geometric.nn import SAGEConv

    PYG_AVAILABLE = True
except ImportError:
    PYG_AVAILABLE = False
    print("torch_geometric not found: using the self-contained GraphSAGE fallback.")


class SAGEConvFallback(nn.Module):
    """Self-contained SAGEConv with mean aggregation (Phase 3 fallback)."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.lin_l = nn.Linear(in_channels, out_channels, bias=True)
        self.lin_r = nn.Linear(in_channels, out_channels, bias=False)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin_l.weight)
        nn.init.zeros_(self.lin_l.bias)
        nn.init.xavier_uniform_(self.lin_r.weight)

    def forward(self, x, edge_index):
        src, dst = edge_index[0], edge_index[1]
        deg = torch.bincount(dst, minlength=x.size(0)).float().clamp(min=1.0).unsqueeze(1)
        out = torch.zeros_like(x)
        out.index_add_(0, dst, x[src])
        return self.lin_l(x) + self.lin_r(out / deg)


class GraphSAGENet(nn.Module):
    """2-layer GraphSAGE for the homogeneous transaction graph (Phase 3)."""

    def __init__(self, in_channels=165, hidden_channels=128, out_channels=1, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        if PYG_AVAILABLE:
            self.conv1 = SAGEConv(in_channels, hidden_channels, aggr="mean")
            self.conv2 = SAGEConv(hidden_channels, out_channels, aggr="mean")
        else:
            self.conv1 = SAGEConvFallback(in_channels, hidden_channels)
            self.conv2 = SAGEConvFallback(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)


graph_sage = GraphSAGENet(165, 128, 1, 0.3)
graph_sage.load_state_dict(
    torch.load(os.path.join(RESULTS_DIR, "GraphSage", "graphsage_model.pt"), map_location="cpu")
)
graph_sage.eval()
print(f"GraphSAGE restored ({sum(p.numel() for p in graph_sage.parameters()):,} params)")


class RelationalConvLayer(nn.Module):
    """One relational mean-aggregation layer over the 4 relations (Phase 4/5)."""

    def __init__(self, in_dims, out_dims):
        super().__init__()
        self.self_tx = nn.Linear(in_dims["tx"], out_dims["tx"], bias=True)
        self.self_addr = nn.Linear(in_dims["addr"], out_dims["addr"], bias=True)
        self.w_tx_to_tx = nn.Linear(in_dims["tx"], out_dims["tx"], bias=False)
        self.w_addr_to_tx = nn.Linear(in_dims["addr"], out_dims["tx"], bias=False)
        self.w_tx_to_addr = nn.Linear(in_dims["tx"], out_dims["addr"], bias=False)
        self.w_addr_to_addr = nn.Linear(in_dims["addr"], out_dims["addr"], bias=False)
        self.reset_parameters()

    def reset_parameters(self):
        for module in (
            self.self_tx,
            self.self_addr,
            self.w_tx_to_tx,
            self.w_addr_to_tx,
            self.w_tx_to_addr,
            self.w_addr_to_addr,
        ):
            if module.weight is not None:
                nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def _mean_aggregate(self, x_src, edge_index, num_dst):
        src, dst = edge_index[0], edge_index[1]
        degree = torch.bincount(dst, minlength=num_dst).float().clamp(min=1.0).unsqueeze(1)
        aggregated = torch.zeros((num_dst, x_src.size(1)), dtype=x_src.dtype, device=x_src.device)
        aggregated.index_add_(0, dst, x_src[src])
        return aggregated / degree

    def forward(self, x_dict, edge_dict):
        x_tx, x_addr = x_dict["tx"], x_dict["addr"]
        num_tx, num_addr = x_tx.size(0), x_addr.size(0)
        out_tx = (
            self.self_tx(x_tx)
            + self.w_tx_to_tx(self._mean_aggregate(x_tx, edge_dict["tx_to_tx"], num_tx))
            + self.w_addr_to_tx(self._mean_aggregate(x_addr, edge_dict["addr_to_tx"], num_tx))
        )
        out_addr = (
            self.self_addr(x_addr)
            + self.w_tx_to_addr(self._mean_aggregate(x_tx, edge_dict["tx_to_addr"], num_addr))
            + self.w_addr_to_addr(self._mean_aggregate(x_addr, edge_dict["addr_to_addr"], num_addr))
        )
        return {"tx": out_tx, "addr": out_addr}


class HeteroRGCN(nn.Module):
    """2-layer relational GCN for heterogeneous fraud detection (Phase 4/5)."""

    def __init__(self, tx_in_dim=165, addr_in_dim=55, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = RelationalConvLayer(
            {"tx": tx_in_dim, "addr": addr_in_dim}, {"tx": hidden_dim, "addr": hidden_dim}
        )
        self.conv2 = RelationalConvLayer(
            {"tx": hidden_dim, "addr": hidden_dim}, {"tx": 1, "addr": hidden_dim}
        )

    def forward(self, x_dict, edge_dict):
        h1 = self.conv1(x_dict, edge_dict)
        h1_tx = F.dropout(F.relu(h1["tx"]), p=self.dropout, training=self.training)
        h1_addr = F.dropout(F.relu(h1["addr"]), p=self.dropout, training=self.training)
        return self.conv2({"tx": h1_tx, "addr": h1_addr}, edge_dict)["tx"]


# The Phase-5 static checkpoint reproduces predictions.csv's `static_score`.
rgcn = HeteroRGCN(165, 55, 128, 0.3)
rgcn.load_state_dict(
    torch.load(
        os.path.join(RESULTS_DIR, "temporal_inductive", "rgcn_static_model.pt"), map_location="cpu"
    )
)
rgcn.eval()
print(f"HeteroRGCN restored ({sum(p.numel() for p in rgcn.parameters()):,} params)")

## 5. Homogeneous Explainability (GraphSAGE)
We explain why GraphSAGE made specific predictions globally and locally.
Unlike XGBoost which relies heavily on aggregate tabular features, GraphSAGE acts explicitly on structure.

To adhere strictly to temporal boundaries and RAM constraints, we construct isolated ego-networks (k-hop subgraphs up to $t$) around the query nodes using standard PyTorch Geometric extractors.

In [ ]:
from sklearn.preprocessing import StandardScaler
from torch_geometric.explain import CaptumExplainer, Explainer
from torch_geometric.utils import k_hop_subgraph

GS_THRESHOLD = 0.83  # frozen Phase-3 operating point

# Homogeneous graph: Phase 3 used every txs_edgelist edge (all 234,355 are intra-step).
tx_edges = pd.read_csv(os.path.join(data_dir, "txs_edgelist.csv"))
u = tx_edges["txId1"].map(node_to_idx).to_numpy()
v = tx_edges["txId2"].map(node_to_idx).to_numpy()
edge_index_sage = torch.tensor(np.stack([u, v]).astype(np.int64))

# Rebuild the exact Phase-3 feature matrix: StandardScaler fit on labelled rows with step <= 34.
tx_raw = pd.read_csv(
    os.path.join(data_dir, "txs_features.csv"), usecols=["txId", "Time step"] + xgb_features
).set_index("txId")
tx_raw = tx_raw.loc[features_df["txId"].values]
raw_tx_arr = tx_raw[xgb_features].to_numpy(dtype="float32")
tx_class = classes_df.set_index("txId")["class"].reindex(features_df["txId"]).to_numpy()
labeled = np.isin(tx_class, [1, 2])
scaler_tx = StandardScaler().fit(raw_tx_arr[labeled & (tx_raw["Time step"].to_numpy() <= 34)])
x_sage = torch.from_numpy(scaler_tx.transform(raw_tx_arr).astype("float32"))

gs_early = gs_preds[(gs_preds["time_step"] >= 35) & (gs_preds["time_step"] <= 42)]
gs_tp = gs_early[(gs_early["y_true"] == 1) & (gs_early["graphsage_score"] >= GS_THRESHOLD)].index

if len(gs_tp) == 0:
    print("No GraphSAGE true positives in the early window; nothing to explain.")
else:
    target_tx = int(gs_tp[0])
    target_idx = node_to_idx[target_tx]

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        target_idx, num_hops=2, edge_index=edge_index_sage, relabel_nodes=True
    )
    x_sub = x_sage[subset]
    with torch.no_grad():
        prob = torch.sigmoid(graph_sage(x_sub, edge_index_sub)[mapping]).item()
    print(
        f"GraphSAGE TP txId {target_tx}: {subset.size(0)} nodes / "
        f"{edge_index_sub.size(1)} edges, p={prob:.4f}"
    )

    explainer = Explainer(
        model=graph_sage,
        algorithm=CaptumExplainer("IntegratedGradients"),
        explanation_type="model",
        node_mask_type="attributes",
        edge_mask_type="object",
        model_config=dict(mode="binary_classification", task_level="node", return_type="raw"),
    )
    explanation = explainer(x_sub, edge_index_sub, index=mapping)
    node_feat_importance = explanation.node_mask[mapping].abs().cpu().numpy().reshape(-1)

    print(f"\nTop 10 attributed features for txId {target_tx}:")
    for idx in np.argsort(node_feat_importance)[::-1][:10]:
        print(f" - {xgb_features[idx]}: {node_feat_importance[idx]:.4f}")

    edge_importance = explanation.edge_mask
    if edge_importance is not None:
        top_edges = torch.argsort(edge_importance.abs(), descending=True)[:10].cpu().numpy()
        print("\nTop edges (subgraph node -> node, attribution):")
        for e in top_edges:
            print(
                f" - {edge_index_sub[0, e].item()} -> {edge_index_sub[1, e].item()}: "
                f"{edge_importance[e].item():+.4f}"
            )

## 6. Heterogeneous Relation Explanations (HeteroRGCN)
Because standard deep explainers like Captum have complexity limits on multi-relational graphs out-of-the-box, we perform **Relation Ablation & Perturbation Analysis**.
By analyzing how the logits shift when specific edge types (`tx_to_tx`, `addr_to_addr`, etc.) are dropped or scaled at test-time for a target transaction, we can isolate exactly *which* heterogeneous pathways drive Phase 5's predictions (e.g. why performance collapsed on `seen` addresses).

In [ ]:
# Build the Phase-4/5 heterogeneous graph. Edge availability is masked per time step, so a node at
# step t is only ever explained with edges and wallet snapshots available at <= t.
MAX_T = 42
EDGE_KEYS = ["tx_to_tx", "addr_to_tx", "tx_to_addr", "addr_to_addr"]

# Address universe and deduplicated (address, time step) wallet snapshots, per the Phase-1 rule.
wallets_classes = pd.read_csv(os.path.join(data_dir, "wallets_classes.csv"))
addr_to_idx = {a: i for i, a in enumerate(wallets_classes["address"].astype(str).values)}
N_ADDR = len(addr_to_idx)

wallet_cols = pd.read_csv(os.path.join(data_dir, "wallets_features.csv"), nrows=0).columns.tolist()
wallet_feature_cols = [c for c in wallet_cols if c not in ("address", "Time step")]
wallets = pd.read_csv(os.path.join(data_dir, "wallets_features.csv"))
wallets = wallets.drop_duplicates(subset=["address", "Time step"]).reset_index(drop=True)
print(
    f"Unique addresses: {N_ADDR:,} | deduplicated snapshots: {len(wallets):,} | "
    f"features: {len(wallet_feature_cols)}"
)

wallets_arr = wallets[wallet_feature_cols].to_numpy(dtype="float32")
wallets_step = wallets["Time step"].to_numpy(dtype="int16")
wallets_addr = wallets["address"].astype(str).map(addr_to_idx).to_numpy(dtype="int64")

# The four directed relations, using the real Elliptic++ column names.
e_tx_tx = pd.read_csv(os.path.join(data_dir, "txs_edgelist.csv"))
e_addr_tx = pd.read_csv(os.path.join(data_dir, "AddrTx_edgelist.csv"))
e_tx_addr = pd.read_csv(os.path.join(data_dir, "TxAddr_edgelist.csv"))
e_addr_addr = pd.read_csv(os.path.join(data_dir, "AddrAddr_edgelist.csv"))

tx_step = time_steps.astype("int16")
endpoints = {
    "tx_to_tx": (
        e_tx_tx["txId1"].map(node_to_idx).to_numpy("int64"),
        e_tx_tx["txId2"].map(node_to_idx).to_numpy("int64"),
    ),
    "addr_to_tx": (
        e_addr_tx["input_address"].astype(str).map(addr_to_idx).to_numpy("int64"),
        e_addr_tx["txId"].map(node_to_idx).to_numpy("int64"),
    ),
    "tx_to_addr": (
        e_tx_addr["txId"].map(node_to_idx).to_numpy("int64"),
        e_tx_addr["output_address"].astype(str).map(addr_to_idx).to_numpy("int64"),
    ),
    "addr_to_addr": (
        e_addr_addr["input_address"].astype(str).map(addr_to_idx).to_numpy("int64"),
        e_addr_addr["output_address"].astype(str).map(addr_to_idx).to_numpy("int64"),
    ),
}

# First-seen step per address: wallet snapshots plus funding/destination edge endpoints.
first_seen = np.full(N_ADDR, np.int16(50))
np.minimum.at(first_seen, wallets_addr, wallets_step)
np.minimum.at(first_seen, endpoints["addr_to_tx"][0], tx_step[endpoints["addr_to_tx"][1]])
np.minimum.at(first_seen, endpoints["tx_to_addr"][1], tx_step[endpoints["tx_to_addr"][0]])

edge_avail = {
    "tx_to_tx": tx_step[endpoints["tx_to_tx"][0]],
    "addr_to_tx": tx_step[endpoints["addr_to_tx"][1]],
    "tx_to_addr": tx_step[endpoints["tx_to_addr"][0]],
    "addr_to_addr": np.maximum(
        first_seen[endpoints["addr_to_addr"][0]], first_seen[endpoints["addr_to_addr"][1]]
    ),
}


def edges_up_to(cutoff):
    """Every relation restricted to edges available at step <= cutoff (no future edges)."""
    out = {}
    for key, (src, dst) in endpoints.items():
        keep = edge_avail[key] <= cutoff
        out[key] = torch.from_numpy(np.stack([src[keep], dst[keep]]).astype("int64"))
    return out


# Transaction features scaled exactly as Phase 4/5: StandardScaler fit on labelled rows <= 34.
tx_raw = pd.read_csv(
    os.path.join(data_dir, "txs_features.csv"), usecols=["txId", "Time step"] + xgb_features
).set_index("txId")
tx_raw = tx_raw.loc[features_df["txId"].values]
raw_tx_arr = tx_raw[xgb_features].to_numpy(dtype="float32")
tx_class = classes_df.set_index("txId")["class"].reindex(features_df["txId"]).to_numpy()
is_labeled = np.isin(tx_class, [1, 2])
scaler_tx_static = StandardScaler().fit(
    raw_tx_arr[is_labeled & (tx_raw["Time step"].to_numpy() <= 34)]
)
X_tx = torch.from_numpy(scaler_tx_static.transform(raw_tx_arr).astype("float32"))


def raw_address_matrix(cutoff):
    """Mean historical wallet features per address, using snapshots up to `cutoff` only."""
    hist = wallets_step <= cutoff
    hist_addr, hist_feat = wallets_addr[hist], wallets_arr[hist]
    sums = np.zeros((N_ADDR, hist_feat.shape[1]), dtype="float32")
    counts = np.zeros(N_ADDR, dtype="float32")
    np.add.at(sums, hist_addr, hist_feat)
    np.add.at(counts, hist_addr, 1.0)
    valid = counts > 0
    sums[valid] /= counts[valid, None]
    sums[~valid] = hist_feat.mean(axis=0) if hist_addr.size else 0.0
    return sums


scaler_addr_static = StandardScaler().fit(raw_address_matrix(34)[first_seen <= 34])


def graph_at_step(step):
    """x_dict / edge_dict for the frozen static model at `step`, reproducing Phase 5 exactly."""
    addr = torch.from_numpy(
        scaler_addr_static.transform(raw_address_matrix(step)).astype("float32")
    )
    return {"tx": X_tx, "addr": addr}, edges_up_to(step)


print("Heterogeneous graph helpers ready (scalers fit on data <= 34; edges masked per step).")

In [ ]:
# Reproduction sanity check: the reconstruction must match predictions.csv's `static_score`
# before any ablation result is trusted.
def static_scores_at_own_step(tx_ids):
    """Static-model probability for each txId, evaluated at that transaction's own time step."""
    scores = {}
    steps = sorted({int(rgcn_preds.loc[tx, "time_step"]) for tx in tx_ids})
    for step in steps:
        x_dict, edge_dict = graph_at_step(step)
        with torch.no_grad():
            probs = torch.sigmoid(rgcn(x_dict, edge_dict).squeeze(1)).cpu().numpy()
        for tx in tx_ids:
            if int(rgcn_preds.loc[tx, "time_step"]) == step:
                scores[int(tx)] = float(probs[node_to_idx[int(tx)]])
        del x_dict, edge_dict, probs
        gc.collect()
    return scores


probe_txs = rgcn_preds[
    (rgcn_preds["time_step"] >= 35) & (rgcn_preds["time_step"] <= 42)
].head(200).index.to_list()
recomputed = static_scores_at_own_step(probe_txs)
deltas = [abs(recomputed[int(tx)] - rgcn_preds.loc[tx, "rgcn_score"]) for tx in probe_txs]
print(f"Reconstruction sanity: max |Δ| vs predictions.csv = {max(deltas):.2e} (should be ~0)")

### 6.1 Relation Ablation over Target Transactions
We test the network's structural reliance: if we drop the `addr_to_addr` path, does the TP collapse to FP? Does FN become TP?

In [ ]:
RGCN_THRESHOLD = 0.795  # frozen Phase-5 operating point

rgcn_early = rgcn_preds[(rgcn_preds["time_step"] >= 35) & (rgcn_preds["time_step"] <= 42)]
rgcn_tp = rgcn_early[
    (rgcn_early["y_true"] == 1) & (rgcn_early["rgcn_score"] >= RGCN_THRESHOLD)
].index.to_numpy()

# Demo on the first few true positives; each is evaluated at its own time step, so no future edge
# or future wallet snapshot can enter the explanation.
demo_txs = [int(t) for t in rgcn_tp[:5]]
ablation_rows = []

for tx in demo_txs:
    step = int(rgcn_preds.loc[tx, "time_step"])
    x_dict, edge_dict = graph_at_step(step)
    tx_idx = node_to_idx[tx]

    def score(edges):
        with torch.no_grad():
            return float(torch.sigmoid(rgcn(x_dict, edges)[tx_idx]).item())

    baseline = score(edge_dict)

    no_tx_to_tx = dict(edge_dict)
    no_tx_to_tx["tx_to_tx"] = torch.empty((2, 0), dtype=torch.long)

    no_address = dict(edge_dict)
    for key in ("addr_to_tx", "tx_to_addr", "addr_to_addr"):
        no_address[key] = torch.empty((2, 0), dtype=torch.long)

    ablation_rows.append(
        {
            "txId": tx,
            "time_step": step,
            "baseline_score": baseline,
            "no_tx_to_tx": score(no_tx_to_tx),
            "no_address_context": score(no_address),
        }
    )
    del x_dict, edge_dict
    gc.collect()

ablation_df = pd.DataFrame(
    ablation_rows,
    columns=["txId", "time_step", "baseline_score", "no_tx_to_tx", "no_address_context"],
)
if not ablation_df.empty:
    ablation_df["delta_no_tx_to_tx"] = ablation_df["no_tx_to_tx"] - ablation_df["baseline_score"]
    ablation_df["delta_no_address_context"] = (
        ablation_df["no_address_context"] - ablation_df["baseline_score"]
    )
print(ablation_df.round(4).to_string(index=False))
ablation_df.to_csv(os.path.join(OUT_DIR, "rgcn_ablation_sample.csv"), index=False)

## 7. Conclusions and Sanity Checks
Ensure explanations exist, boundaries are respected, and no NaNs corrupted the attributions.

In [ ]:
checks = []

# 1. Temporal boundary: the newest edge used for a step-t node is available at <= t.
def newest_edge_used(step):
    worst = 0
    for key in EDGE_KEYS:
        keep = edge_avail[key] <= step
        if keep.any():
            worst = max(worst, int(edge_avail[key][keep].max()))
    return worst


boundary_ok = all(newest_edge_used(t) <= t for t in range(35, MAX_T + 1))
checks.append(
    {"metric": "Edges at step t available at <= t", "value": str(boundary_ok), "ok": bool(boundary_ok)}
)

# 2. SHAP covers every model feature.
checks.append(
    {
        "metric": "SHAP attributions for all 165 features",
        "value": str(len(shap_importance)),
        "ok": len(shap_importance) == 165,
    }
)

# 3. Ablation produced finite scores.
score_cols = ["baseline_score", "no_tx_to_tx", "no_address_context"]
finite_ok = bool(np.isfinite(ablation_df[score_cols].to_numpy()).all()) if not ablation_df.empty else False
checks.append({"metric": "Ablation scores finite", "value": str(finite_ok), "ok": finite_ok})

checks_df = pd.DataFrame(checks)
checks_df.to_csv(os.path.join(OUT_DIR, "checks.csv"), index=False)
print(checks_df.to_string(index=False))

with open(os.path.join(OUT_DIR, "explainability_digest.txt"), "w") as f:
    f.write("Phase 6 Explainability Digest\n")
    f.write("-----------------------------\n")
    f.write("1. XGBoost (frozen, 165 features) attributions are dominated by the handcrafted Local/Aggregate block.\n")
    f.write("2. GraphSAGE attributions are confined to intra-step neighbourhoods (the homogeneous graph has no cross-step edges).\n")
    f.write("3. HeteroRGCN relation ablations are evaluated at each target's own time step, so no future edges or wallet snapshots leak in.\n")